In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

sys.path.append(os.path.abspath('../'))

from tagging_system import automate_tagging as auto_tag
from models.custom_classifier import CustomClassifier

In [3]:
import numpy as np
import pandas as pd

In [4]:
books = pd.read_csv('../data/data_with_author_and_awards.csv',
dtype = {
    'isbn' : 'str', # do this explicitly to avoide getting a warning by the interpreter
    'author_birthyear' : 'Int64', # we have to explicitly do this to avoid pandas implicitly casting as float
    'title_id' : 'Int64'
},
)
books = books.dropna() # we drop the books without descriptions because these are very, very unlikely to be winners any

In [5]:
tags = auto_tag.load_tags()
transformer = auto_tag.load_transformer()
tags_encoded = auto_tag.encode_tags()
tags_map = auto_tag.tag_dictionary()

In [6]:
books = auto_tag.encode_books(books, transformer) # takes a while

In [7]:
books['target'] = books.hugo | books.locus
books['num_prev_awards'] = books.Hugo_Awards_Previously + books.Locus_Awards_Previously

In [8]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score

In [9]:
# manually train test split our data

books_train = books[books.release_year <= 2014]
books_test = books[books.release_year <= 2014]

books_tt = books_train[books_train.release_year <= 2005]
books_val = books_train[books_train.release_year > 2005]

In [10]:
X_tt = books_train[['num_prev_awards', 'encoded_synopsis']]
y_tt = books_train['target']

X_val = books_val[['num_prev_awards', 'encoded_synopsis']]
y_val = books_val['target']

In [12]:
my_classifier = CustomClassifier(n_neighbors = 2, weights='distance', n_jobs = -1, class_weight='balanced')

my_classifier.fit(X_tt, y_tt)

In [13]:
pred = my_classifier.predict(X_val)
print('f1', f1_score(y_val, pred))
print('precision', precision_score(y_val, pred))

f1 0.977859778597786
precision 0.9636363636363636


These are some suspiciously good metrics...

In [14]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_val, pred, normalize='all')

array([[9.57528366e-01, 1.53327200e-03],
       [3.06654400e-04, 4.06317081e-02]])

It looks like what's happening is that it's making any false negatives or false positives, but have a decently high number of true positives, and a very high rate of true negatives.

In [15]:
tt_cv = [ books_train[(1971 <= books_train.release_year) & (books_train.release_year <= 1977)] ]
val_cv = [ books_train[(1978 <= books_train.release_year) & (books_train.release_year <= 1979)] ]

tt_cv.append( books_train[(1971 <= books_train.release_year) & (books_train.release_year <= 1985)] )
val_cv.append( books_train[(1986 <= books_train.release_year) & (books_train.release_year <= 1987)] )

tt_cv.append( books_train[(1971 <= books_train.release_year) & (books_train.release_year <= 1993)] )
val_cv.append( books_train[(1994 <= books_train.release_year) & (books_train.release_year <= 1995)] )

tt_cv.append( books_train[(1971 <= books_train.release_year) & (books_train.release_year <= 2001)] )
val_cv.append( books_train[(2002 <= books_train.release_year) & (books_train.release_year <= 2003)] )

tt_cv.append( books_train[(1971 <= books_train.release_year) & (books_train.release_year <= 2011)] )
val_cv.append( books_train[(2012 <= books_train.release_year) & (books_train.release_year <= 2014)] )

In [19]:
f1s = dict()
precs = dict()

my_classifier = CustomClassifier(n_neighbors = 2, weights='distance', n_jobs = -1, class_weight='balanced')

for i in range(5):
    X_tt_cv = tt_cv[i][['num_prev_awards', 'encoded_synopsis']]
    y_tt_cv = tt_cv[i].target

    X_ho = val_cv[i][['num_prev_awards', 'encoded_synopsis']]
    y_ho = val_cv[i].target

    my_classifier.fit(X_tt_cv, y_tt_cv)

    pred = my_classifier.predict(X_ho)

    f1s['Fold {} F_1'.format(i)] = f1_score(y_ho, pred)
    precs['Fold {} Precision'.format(i)] = precision_score(y_ho, pred)
    

In [20]:
f1s

{'Fold 0 F_1': 0.21621621621621623,
 'Fold 1 F_1': 0.2204724409448819,
 'Fold 2 F_1': 0.23841059602649006,
 'Fold 3 F_1': 0.16049382716049382,
 'Fold 4 F_1': 0.17424242424242425}

In [21]:
precs

{'Fold 0 Precision': 0.2222222222222222,
 'Fold 1 Precision': 0.20588235294117646,
 'Fold 2 Precision': 0.1875,
 'Fold 3 Precision': 0.12380952380952381,
 'Fold 4 Precision': 0.14285714285714285}

As expected, we get much lower scores when we do cross-validation. This makes me wonder what was happening when we did the fitting on books_tt to get such ridiculously high scores.

In [ ]:
from scipy import stats

class CustomBaggingClassifier:
    def __init__(self, base_estimator = CustomClassifier, n_estimators = 10, kwargs = {}):
        '''
        (Mostly) copied from the ensemble_i problem session
        Parameters:
            base_estimator: Our CustomClassifier class
            n_estimators: Number of estimators in the ensemble
            kwargs: A dictionary of keyword arguments to pass to our base_estimator
        Attributes:
            self.estimators: A list of instantiated base estimators
        '''

        self.kwargs = kwargs
        self.n_estimators = n_estimators
        self.estimators = [base_estimator(**kwargs) for _ in range(n_estimators)]
    
    def fit(self, X, y):
        '''
        Inputs:
            X: DataFrame that is specified with the same columns as taken in by our CustomClassifier
            y: Target of Hugo | Locus award nominees
        '''
        rng = np.random.default_rng()
        n_samples = X.shape[0]

        for estimator in self.estimators:
            indices = rng.choice(n_samples, n_samples, replace = True)
            X_boot = X.iloc[indices]
            y_boot = y.iloc[indices]

            estimator.fit(X_boot, y_boot)
        return
    
    def predict(self, X):
        preds = np.array([estimator.predict(X) for estimator in self.estimators])
        preds = np.array([
            bool(np.argmax(np.bincount(preds[i]))) for i in range(len(preds))
        ])

        return preds

In [43]:
n_ests = [2, 5, 10, 100]

avg_f1s = dict()
avg_precs = dict()

kwargs = {'n_neighbors' : 2, 'weights' : 'distance', 'n_jobs' : -1, 'class_weight' :'balanced'}


for n in n_ests:
    avg_f1 = 0
    avg_prec = 0
    for i in range(5):
        bag = CustomBaggingClassifier(
            base_estimator = CustomClassifier,
            n_estimators = n,
            kwargs = kwargs
        )

        X_tt_cv = tt_cv[i][['num_prev_awards', 'encoded_synopsis']]
        y_tt_cv = tt_cv[i].target

        X_ho = val_cv[i][['num_prev_awards', 'encoded_synopsis']]
        y_ho = val_cv[i].target

        bag.fit(X_tt_cv, y_tt_cv)
        pred = bag.predict(X_ho)

        avg_f1 += f1_score(y_ho, pred)
        avg_prec += precision_score(y_ho, pred)

    avg_f1s['{} estimators'.format(n)] = avg_f1 / 5
    avg_precs['{} estimators'.format(n)] = avg_prec / 5

TypeError: 'int' object is not iterable

In [ ]:
avg_f1s